In [1]:
from pathlib import Path

DATA_DIR = Path("output")

OUTPUT_DIR = Path("model_outputs")

RANDOM_STATE = 42

In [2]:
WHEN_CSV_PATTERN = "dataset_winsize*_when.csv"


def convert_when_to_where_dataset_path(when_dataset_path):
    """Return the matching where-dataset path for a when-dataset CSV path."""
    suffix = "_when.csv"

    return when_dataset_path.with_name(
        f"{when_dataset_path.name[:-len(suffix)]}_where.csv"
    )


when_csv_files = sorted(
    DATA_DIR.glob(WHEN_CSV_PATTERN),
    key=lambda p: float(p.stem.split("winsize", 1)[1].removesuffix("h_when")),
)


when_csv_files

[PosixPath('output/dataset_winsize0.25h_when.csv'),
 PosixPath('output/dataset_winsize0.5h_when.csv'),
 PosixPath('output/dataset_winsize1h_when.csv'),
 PosixPath('output/dataset_winsize2h_when.csv'),
 PosixPath('output/dataset_winsize3h_when.csv'),
 PosixPath('output/dataset_winsize4h_when.csv'),
 PosixPath('output/dataset_winsize5h_when.csv'),
 PosixPath('output/dataset_winsize6h_when.csv'),
 PosixPath('output/dataset_winsize7h_when.csv'),
 PosixPath('output/dataset_winsize8h_when.csv'),
 PosixPath('output/dataset_winsize9h_when.csv'),
 PosixPath('output/dataset_winsize10h_when.csv'),
 PosixPath('output/dataset_winsize11h_when.csv'),
 PosixPath('output/dataset_winsize12h_when.csv')]

In [3]:
import pandas as pd


def load_when_where_datasets(when_dataset_path):
    where_dataset_path = convert_when_to_where_dataset_path(when_dataset_path)

    when_df = pd.read_csv(when_dataset_path)
    where_df = pd.read_csv(where_dataset_path)

    return when_df, where_df


when_df, where_df = load_when_where_datasets(when_csv_files[0])

In [4]:
# check if fold_id columns are equal
all(when_df.fold_id.to_numpy() == where_df.fold_id.to_numpy())

True

In [6]:
import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import expit

In [19]:
# load "t"s
t = when_df["label_time_to_event_seconds"].to_numpy(dtype=float)

T = 24 * 3600.0
mask = np.isfinite(t) & (t >= 0) & (t <= T)
t_fit = t[mask]

print("Number of samples inside {T/3600.0}hours:", len(t_fit))
print("Number of samples outside {T/3600.0}hours:", len(t) - len(t_fit))

Number of samples inside {T/3600.0}hours: 5950
Number of samples outside {T/3600.0}hours: 33
